In [1]:
##% [Cell 1] Import Libraries and Set Up Logging
import os
import pandas as pd
import numpy as np
import datetime
import warnings
import matplotlib.pyplot as plt
import time
import optuna
from optuna.samplers import TPESampler
from skopt import BayesSearchCV
from skopt.space import Real, Integer, Categorical

from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.impute import KNNImputer
from sklearn.model_selection import KFold, cross_val_score, train_test_split
from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectFromModel, RFE, RFECV
from sklearn.decomposition import PCA
import xgboost as xgb
import lightgbm as lgb
import catboost as cb
import joblib

# Suppress warnings
warnings.filterwarnings("ignore")

# Start timer to track total execution time
start_time = time.time()

# Base directories
base_dir = r"/kaggle/input/eyds-1503-dataset"
submission_dir = r"/kaggle/working/"
os.makedirs(submission_dir, exist_ok=True)
timestamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')

# Create a log file in the submission directory
log_file = os.path.join(submission_dir, f"model_training_log_{timestamp}.txt")

def log_message(message):
    """Log message to both console and log file (for debugging purposes)."""
    print(message)
    with open(log_file, 'a') as f:
        f.write(message + '\n')

log_message(f"=== Starting ML Pipeline at {timestamp} ===")

=== Starting ML Pipeline at 20250320_002028 ===


In [2]:
##% [Cell 2] Load Combined Datasets
# Load imputed training and validation datasets (ensure these files exist from previous imputation step)
train_csv = os.path.join(base_dir, "Training_data_1603.csv")
train_df = pd.read_csv(train_csv)
log_message(f"Combined training data loaded with shape: {train_df.shape}")

val_csv = os.path.join(base_dir, "Validation_data_1603.csv")
val_df = pd.read_csv(val_csv)
log_message(f"Combined validation data loaded with shape: {val_df.shape}")

Combined training data loaded with shape: (11229, 270)
Combined validation data loaded with shape: (1040, 269)


In [3]:
##% [Cell 3] Prepare Model-Ready Data
# Columns to drop from modeling (non-model columns)
cols_to_drop = ['Latitude', 'Longitude', 'datetime']

# For training data: drop non-model columns and cast numeric columns to float64
train_model_df = train_df.drop(columns=[col for col in cols_to_drop if col in train_df.columns])
log_message(f"Training model data shape after dropping non-model columns: {train_model_df.shape}")
numeric_cols = train_model_df.select_dtypes(include=[np.number]).columns
train_model_df[numeric_cols] = train_model_df[numeric_cols].astype(np.float64)

# Split features and target
X = train_model_df.drop(columns=['UHI Index'])
y = train_model_df['UHI Index']
log_message(f"X shape: {X.shape}, y shape: {y.shape}")

# Prepare validation data similarly
val_model_df = val_df.drop(columns=[col for col in cols_to_drop if col in val_df.columns])
log_message(f"Validation model data shape after dropping non-model columns: {val_model_df.shape}")
numeric_cols_val = val_model_df.select_dtypes(include=[np.number]).columns
val_model_df[numeric_cols_val] = val_model_df[numeric_cols_val].astype(np.float64)
X_val = val_model_df.drop(columns=['UHI Index'], errors='ignore')
log_message(f"X_val shape: {X_val.shape}")

Training model data shape after dropping non-model columns: (11229, 267)
X shape: (11229, 266), y shape: (11229,)
Validation model data shape after dropping non-model columns: (1040, 267)
X_val shape: (1040, 266)


In [ ]:
##% [Cell 4] Advanced Imputation and Feature Engineering
log_message("\n=== Imputation and Feature Engineering ===")
# Drop columns with >80% missing values
missing_threshold = 0.5
high_missing_cols = [col for col in X.columns if X[col].isnull().mean() > missing_threshold]
if high_missing_cols:
    log_message(f"Dropping {len(high_missing_cols)} columns with >80% missing values: {high_missing_cols}")
    X = X.drop(columns=high_missing_cols)
    if set(high_missing_cols).issubset(set(X_val.columns)):
        X_val = X_val.drop(columns=high_missing_cols)
else:
    log_message("No columns with >80% missing values to drop.")

# Ensure validation data has the same columns as training data
X_val = X_val[X.columns]
log_message(f"X shape after aligning validation data: {X.shape}, X_val shape: {X_val.shape}")

# Advanced imputation using KNNImputer (n_neighbors=5)
log_message("\nPerforming KNN imputation on training and validation data...")
imputer = KNNImputer(n_neighbors=5)
X_imputed = pd.DataFrame(imputer.fit_transform(X), columns=X.columns)
X_val_imputed = pd.DataFrame(imputer.transform(X_val), columns=X_val.columns)
log_message(f"X_imputed shape: {X_imputed.shape}, X_val_imputed shape: {X_val_imputed.shape}")

# Replace any remaining NaN with 0 (debug: check for remaining missing values)
X_imputed = X_imputed.fillna(0)
X_val_imputed = X_val_imputed.fillna(0)
log_message(f"Missing values in X_imputed: {X_imputed.isnull().sum().sum()}")

# Creating interaction features from a subset of environmental variables (limit to 5 to control feature explosion)
log_message("\nCreating interaction features...")
interaction_features = []
env_features = [col for col in X_imputed.columns if any(term in col.lower() for term in 
               ['temp', 'uhi', 'building', 'water', 'tree', 'park'])][:5]

if len(env_features) >= 2:
    for i in range(len(env_features)):
        for j in range(i+1, len(env_features)):
            feat_name = f"interaction_{env_features[i]}_{env_features[j]}"
            X_imputed[feat_name] = X_imputed[env_features[i]] * X_imputed[env_features[j]]
            X_val_imputed[feat_name] = X_val_imputed[env_features[i]] * X_val_imputed[env_features[j]]
            interaction_features.append(feat_name)
    log_message(f"Created {len(interaction_features)} interaction features: {interaction_features}")
else:
    log_message("Not enough environmental features for interaction terms.")

# Save feature statistics for debugging/analysis
X_stats = pd.DataFrame({
    'mean': X_imputed.mean(),
    'std': X_imputed.std(),
    'missing_pct': X.isnull().mean()
})
X_stats_file = os.path.join(submission_dir, f"feature_stats_{timestamp}.csv")
X_stats.to_csv(X_stats_file)
log_message(f"Feature statistics saved to {X_stats_file}")


=== Imputation and Feature Engineering ===
Dropping 32 columns with >80% missing values: ['S02_avg_JJA', 'S02_median_JJA', 'S02_std_JJA', 'S02_min_JJA', 'S02_max_JJA', 'S02_p25_JJA', 'S02_p75_JJA', 'S02_val_21Jul2021', 'Ozone_avg_JJA', 'Ozone_median_JJA', 'Ozone_std_JJA', 'Ozone_min_JJA', 'Ozone_max_JJA', 'Ozone_p25_JJA', 'Ozone_p75_JJA', 'Ozone_val_21Jul2021', 'CO_avg_JJA', 'CO_median_JJA', 'CO_std_JJA', 'CO_min_JJA', 'CO_max_JJA', 'CO_p25_JJA', 'CO_p75_JJA', 'CO_val_21Jul2021', 'NO2_avg_JJA', 'NO2_median_JJA', 'NO2_std_JJA', 'NO2_min_JJA', 'NO2_max_JJA', 'NO2_p25_JJA', 'NO2_p75_JJA', 'NO2_val_21Jul2021']
X shape after aligning validation data: (11229, 234), X_val shape: (1040, 234)

Performing KNN imputation on training and validation data...
X_imputed shape: (11229, 234), X_val_imputed shape: (1040, 234)
Missing values in X_imputed: 0

Creating interaction features...
Created 10 interaction features: ['interaction_building_count_10m_building_count_20m', 'interaction_building_count_

In [5]:
##% [Cell 5] Feature Selection
log_message("\n=== Feature Selection ===")
# Split imputed training data for feature selection evaluation (80-20 split)
X_train, X_valid, y_train, y_valid = train_test_split(
    X_imputed, y, test_size=0.2, random_state=42
)
log_message(f"After train/valid split: X_train: {X_train.shape}, X_valid: {X_valid.shape}")

# Initial feature selection using ExtraTrees
log_message("\nPerforming initial feature selection with ExtraTrees...")
selector = SelectFromModel(
    ExtraTreesRegressor(n_estimators=100, random_state=42),
    threshold='median'
)
selector.fit(X_train, y_train)
selected_features = X_train.columns[selector.get_support()].tolist()
log_message(f"Selected {len(selected_features)} features out of {X_train.shape[1]}: {selected_features}")

# Apply feature selection
X_train_selected = selector.transform(X_train)
X_valid_selected = selector.transform(X_valid)
X_val_selected = selector.transform(X_val_imputed)
log_message(f"X_train_selected shape: {X_train_selected.shape}")

# Convert back to DataFrames with feature names
X_train_selected = pd.DataFrame(X_train_selected, columns=selected_features)
X_valid_selected = pd.DataFrame(X_valid_selected, columns=selected_features)
X_val_selected = pd.DataFrame(X_val_selected, columns=selected_features)

# Save selected features for debugging
selected_features_file = os.path.join(submission_dir, f"selected_features_{timestamp}.csv")
pd.DataFrame({'selected_features': selected_features}).to_csv(selected_features_file, index=False)
log_message(f"Selected features saved to {selected_features_file}")


=== Feature Selection ===
After train/valid split: X_train: (8983, 244), X_valid: (2246, 244)

Performing initial feature selection with ExtraTrees...
Selected 122 features out of 244: ['building_count_300m', 'building_count_400m', 'building_count_500m', 'building_count_750m', 'building_count_1000m', 'dist_to_road', 'dist_to_park', 'dist_to_water', 'roads_ratio_100m', 'roads_ratio_250m', 'roads_ratio_500m', 'roads_ratio_1000m', 'parks_ratio_100m', 'parks_ratio_250m', 'parks_ratio_500m', 'parks_ratio_1000m', 'parks_weighted_score', 'tree_count_100m', 'tree_count_200m', 'tree_avg_diam_200m', 'tree_count_300m', 'tree_avg_diam_300m', 'tree_count_500m', 'tree_avg_diam_500m', 'tree_count_750m', 'tree_avg_diam_750m', 'tree_count_1000m', 'tree_avg_diam_1000m', 'Fine particles (PM 2.5)', 'Nitrogen dioxide (NO2)', 'Ozone (O3)', 'tower_count_200m', 'tower_count_500m', 'tower_count_1000m', 'sum_net_emissions_mtco2e_500m', 'sum_weather_normalized_site_energy_use_(kbtu)_500m', 'sum_weather_normaliz

In [6]:
##% [Cell 6] Hyperparameter Optimization with Optuna for ExtraTrees
log_message("\n=== Hyperparameter Optimization with Optuna for ExtraTrees ===")
def objective_et(trial):
    """Optuna objective function for ExtraTrees."""
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 500, step=50),
        'max_depth': trial.suggest_int('max_depth', 10, 50, step=2),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 10),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 4),
        'max_features': trial.suggest_float('max_features', 0.1, 1.0),
        'bootstrap': trial.suggest_categorical('bootstrap', [True, False]),
        'random_state': 42,
        'n_jobs': -1
    }
    
    model = ExtraTreesRegressor(**params)
    cv = KFold(n_splits=5, shuffle=True, random_state=42)
    cv_scores = cross_val_score(model, X_train_selected, y_train, cv=cv, scoring='r2', n_jobs=-1)
    return cv_scores.mean()

study_et = optuna.create_study(direction='maximize', sampler=TPESampler(seed=42), pruner=optuna.pruners.MedianPruner())
study_et.optimize(objective_et, n_trials=200, show_progress_bar=True)

best_et_params = study_et.best_params
best_et_params['random_state'] = 42
best_et_params['n_jobs'] = -1
log_message(f"\nBest ExtraTrees Parameters: {best_et_params}")
log_message(f"Best ExtraTrees CV R² Score: {study_et.best_value:.6f}")

# Train ExtraTrees model with best parameters
et_model = ExtraTreesRegressor(**best_et_params)
et_model.fit(X_train_selected, y_train)
y_valid_et_pred = et_model.predict(X_valid_selected)
et_valid_r2 = r2_score(y_valid, y_valid_et_pred)
et_valid_rmse = np.sqrt(mean_squared_error(y_valid, y_valid_et_pred))
log_message(f"ExtraTrees Validation R²: {et_valid_r2:.6f}, RMSE: {et_valid_rmse:.6f}")

[I 2025-03-20 00:22:39,594] A new study created in memory with name: no-name-61ae221c-7979-410a-9551-a028dd514156



=== Hyperparameter Optimization with Optuna for ExtraTrees ===


  0%|          | 0/200 [00:00<?, ?it/s]

[I 2025-03-20 00:22:52,352] Trial 0 finished with value: 0.9526036256004214 and parameters: {'n_estimators': 250, 'max_depth': 48, 'min_samples_split': 8, 'min_samples_leaf': 3, 'max_features': 0.24041677639819287, 'bootstrap': True}. Best is trial 0 with value: 0.9526036256004214.
[I 2025-03-20 00:23:57,824] Trial 1 finished with value: 0.9632428265632476 and parameters: {'n_estimators': 450, 'max_depth': 34, 'min_samples_split': 8, 'min_samples_leaf': 1, 'max_features': 0.9729188669457949, 'bootstrap': True}. Best is trial 1 with value: 0.9632428265632476.
[I 2025-03-20 00:24:13,689] Trial 2 finished with value: 0.9615469484290303 and parameters: {'n_estimators': 150, 'max_depth': 16, 'min_samples_split': 4, 'min_samples_leaf': 3, 'max_features': 0.48875051677790415, 'bootstrap': False}. Best is trial 1 with value: 0.9632428265632476.
[I 2025-03-20 00:24:42,870] Trial 3 finished with value: 0.9693790411385625 and parameters: {'n_estimators': 150, 'max_depth': 22, 'min_samples_split':

In [7]:
##% [Cell 7] Hyperparameter Optimization with Optuna for XGBoost
log_message("\n=== Hyperparameter Optimization with Optuna for XGBoost ===")
def objective_xgb(trial):
    """Optuna objective function for XGBoost."""
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 500, step=50),
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        'random_state': 42
    }
    
    model = xgb.XGBRegressor(**params)
    cv = KFold(n_splits=5, shuffle=True, random_state=42)
    cv_scores = cross_val_score(model, X_train_selected, y_train, cv=cv, scoring='r2', n_jobs=-1)
    return cv_scores.mean()

study_xgb = optuna.create_study(direction='maximize', sampler=TPESampler(seed=42), pruner=optuna.pruners.MedianPruner())
study_xgb.optimize(objective_xgb, n_trials=200, show_progress_bar=True)

best_xgb_params = study_xgb.best_params
best_xgb_params['random_state'] = 42
log_message(f"\nBest XGBoost Parameters: {best_xgb_params}")
log_message(f"Best XGBoost CV R² Score: {study_xgb.best_value:.6f}")

# Train XGBoost model with best parameters
xgb_model = xgb.XGBRegressor(**best_xgb_params)
xgb_model.fit(X_train_selected, y_train)
y_valid_xgb_pred = xgb_model.predict(X_valid_selected)
xgb_valid_r2 = r2_score(y_valid, y_valid_xgb_pred)
xgb_valid_rmse = np.sqrt(mean_squared_error(y_valid, y_valid_xgb_pred))
log_message(f"XGBoost Validation R²: {xgb_valid_r2:.6f}, RMSE: {xgb_valid_rmse:.6f}")

[I 2025-03-20 03:06:30,514] A new study created in memory with name: no-name-7569216d-e805-4521-9a6c-e28e74639826



=== Hyperparameter Optimization with Optuna for XGBoost ===


  0%|          | 0/200 [00:00<?, ?it/s]

[I 2025-03-20 03:06:59,400] Trial 0 finished with value: 0.9678310085050782 and parameters: {'n_estimators': 250, 'max_depth': 12, 'learning_rate': 0.1205712628744377, 'subsample': 0.7993292420985183, 'colsample_bytree': 0.5780093202212182, 'min_child_weight': 2, 'reg_alpha': 3.3323645788192616e-08, 'reg_lambda': 0.6245760287469893}. Best is trial 0 with value: 0.9678310085050782.
[I 2025-03-20 03:09:21,230] Trial 1 finished with value: 0.9612355103999766 and parameters: {'n_estimators': 350, 'max_depth': 10, 'learning_rate': 0.010725209743171997, 'subsample': 0.9849549260809971, 'colsample_bytree': 0.9162213204002109, 'min_child_weight': 3, 'reg_alpha': 4.329370014459266e-07, 'reg_lambda': 4.4734294104626844e-07}. Best is trial 0 with value: 0.9678310085050782.
[I 2025-03-20 03:09:46,491] Trial 2 finished with value: 0.9613232784868156 and parameters: {'n_estimators': 200, 'max_depth': 8, 'learning_rate': 0.04345454109729477, 'subsample': 0.645614570099021, 'colsample_bytree': 0.80592

In [8]:
##% [Cell 8] Bayesian Optimization for LightGBM
log_message("\n=== Bayesian Optimization for LightGBM ===")
lgb_search_spaces = {
    'learning_rate': Real(0.01, 0.3, prior='log-uniform'),
    'n_estimators': Integer(100, 500),
    'num_leaves': Integer(20, 150),
    'max_depth': Integer(3, 12),
    'min_child_samples': Integer(5, 100),
    'subsample': Real(0.5, 1.0),
    'colsample_bytree': Real(0.5, 1.0),
    'reg_alpha': Real(1e-8, 10.0, prior='log-uniform'),
    'reg_lambda': Real(1e-8, 10.0, prior='log-uniform')
}

lgb_model = lgb.LGBMRegressor(random_state=42)
bayes_search = BayesSearchCV(
    lgb_model,
    lgb_search_spaces,
    n_iter=30,
    cv=5,
    scoring='r2',
    random_state=42,
    verbose=0,
    n_jobs=-1
)
log_message("Running Bayesian Optimization for LightGBM...")
bayes_search.fit(X_train_selected, y_train)
best_lgb_params = bayes_search.best_params_
best_lgb_params['random_state'] = 42
log_message(f"\nBest LightGBM Parameters: {best_lgb_params}")
log_message(f"Best LightGBM CV R² Score: {bayes_search.best_score_:.6f}")

# Train LightGBM model with best parameters
lgb_model = lgb.LGBMRegressor(**best_lgb_params)
lgb_model.fit(X_train_selected, y_train)
y_valid_lgb_pred = lgb_model.predict(X_valid_selected)
lgb_valid_r2 = r2_score(y_valid, y_valid_lgb_pred)
lgb_valid_rmse = np.sqrt(mean_squared_error(y_valid, y_valid_lgb_pred))
log_message(f"LightGBM Validation R²: {lgb_valid_r2:.6f}, RMSE: {lgb_valid_rmse:.6f}")


=== Bayesian Optimization for LightGBM ===
Running Bayesian Optimization for LightGBM...
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.007762 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 21093
[LightGBM] [Info] Number of data points in the train set: 8983, number of used features: 122
[LightGBM] [Info] Start training from score 0.999997
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further split

In [9]:
##% [Cell 9] Model Blending
log_message("\n=== Model Blending ===")
models = {
    'ExtraTrees': (et_model, et_valid_r2),
    'XGBoost': (xgb_model, xgb_valid_r2),
    'LightGBM': (lgb_model, lgb_valid_r2)
}
total_score = sum(score for _, score in models.values())
weights = {name: score / total_score for name, (_, score) in models.items()}
log_message(f"Blending weights based on validation R² scores: {weights}")

y_valid_blend = np.zeros(len(y_valid))
for name, (model, _) in models.items():
    y_valid_blend += weights[name] * model.predict(X_valid_selected)
blend_valid_r2 = r2_score(y_valid, y_valid_blend)
blend_valid_rmse = np.sqrt(mean_squared_error(y_valid, y_valid_blend))
log_message(f"Blended Model Validation R²: {blend_valid_r2:.6f}, RMSE: {blend_valid_rmse:.6f}")


=== Model Blending ===
Blending weights based on validation R² scores: {'ExtraTrees': 0.33398140162838386, 'XGBoost': 0.33347993150556315, 'LightGBM': 0.3325386668660531}
Blended Model Validation R²: 0.973082, RMSE: 0.002649


In [10]:
##% [Cell 10] Final Model Training and Prediction
log_message("\n=== Final Model Training and Prediction ===")
if blend_valid_r2 > max(et_valid_r2, xgb_valid_r2, lgb_valid_r2):
    best_approach = "blended"
    best_score = blend_valid_r2
    log_message("Using blended model approach for final predictions")
else:
    best_model, best_score = max(
        [('ExtraTrees', et_valid_r2), ('XGBoost', xgb_valid_r2), ('LightGBM', lgb_valid_r2)],
        key=lambda x: x[1]
    )
    best_approach = best_model
    log_message(f"Using {best_model} for final predictions (R²: {best_score:.6f})")

log_message("Retraining models on full training data...")
et_full_model = ExtraTreesRegressor(**best_et_params)
et_full_model.fit(X_imputed[selected_features], y)
xgb_full_model = xgb.XGBRegressor(**best_xgb_params)
xgb_full_model.fit(X_imputed[selected_features], y)
lgb_full_model = lgb.LGBMRegressor(**best_lgb_params)
lgb_full_model.fit(X_imputed[selected_features], y)

# Save full models for debugging/production
joblib.dump(et_full_model, os.path.join(submission_dir, f"extratrees_model_{timestamp}.pkl"))
joblib.dump(xgb_full_model, os.path.join(submission_dir, f"xgboost_model_{timestamp}.pkl"))
joblib.dump(lgb_full_model, os.path.join(submission_dir, f"lightgbm_model_{timestamp}.pkl"))
log_message("Full models retrained and saved.")

# Generate predictions on validation data using the best approach
if best_approach == "blended":
    val_pred = (
        weights['ExtraTrees'] * et_full_model.predict(X_val_imputed[selected_features]) +
        weights['XGBoost'] * xgb_full_model.predict(X_val_imputed[selected_features]) +
        weights['LightGBM'] * lgb_full_model.predict(X_val_imputed[selected_features])
    )
elif best_approach == "ExtraTrees":
    val_pred = et_full_model.predict(X_val_imputed[selected_features])
elif best_approach == "XGBoost":
    val_pred = xgb_full_model.predict(X_val_imputed[selected_features])
else:  # LightGBM
    val_pred = lgb_full_model.predict(X_val_imputed[selected_features])


=== Final Model Training and Prediction ===
Using ExtraTrees for final predictions (R²: 0.973478)
Retraining models on full training data...
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.009449 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 21146
[LightGBM] [Info] Number of data points in the train set: 11229, number of used features: 122
[LightGBM] [Info] Start training from score 1.000001
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, 

In [11]:
##% [Cell 11] Create and Save Final Submission
submission = val_df[['Latitude', 'Longitude']].copy()
submission['UHI Index'] = val_pred
submission_file = os.path.join(submission_dir, f"{best_approach}_submission_{timestamp}.csv")
submission.to_csv(submission_file, index=False)
log_message(f"Final submission saved to {submission_file}")

# Save best hyperparameters and blending weights for reference
pd.DataFrame([best_et_params]).to_csv(os.path.join(submission_dir, f"extratrees_params_{timestamp}.csv"), index=False)
pd.DataFrame([best_xgb_params]).to_csv(os.path.join(submission_dir, f"xgboost_params_{timestamp}.csv"), index=False)
pd.DataFrame([best_lgb_params]).to_csv(os.path.join(submission_dir, f"lightgbm_params_{timestamp}.csv"), index=False)
pd.DataFrame([weights]).to_csv(os.path.join(submission_dir, f"blend_weights_{timestamp}.csv"), index=False)
log_message("Hyperparameters and blending weights saved.")

Final submission saved to /kaggle/working/ExtraTrees_submission_20250320_002028.csv
Hyperparameters and blending weights saved.


In [12]:
##% [Cell 12] Feature Importance Analysis and Final Logging
log_message("\n=== Feature Importance Analysis ===")
et_importances = pd.DataFrame({
    'feature': selected_features,
    'importance': et_full_model.feature_importances_
}).sort_values('importance', ascending=False)
xgb_importances = pd.DataFrame({
    'feature': selected_features,
    'importance': xgb_full_model.feature_importances_
}).sort_values('importance', ascending=False)
lgb_importances = pd.DataFrame({
    'feature': selected_features,
    'importance': lgb_full_model.feature_importances_
}).sort_values('importance', ascending=False)

# Save feature importances to CSV files
et_importances.to_csv(os.path.join(submission_dir, f"et_feature_importance_{timestamp}.csv"), index=False)
xgb_importances.to_csv(os.path.join(submission_dir, f"xgb_feature_importance_{timestamp}.csv"), index=False)
lgb_importances.to_csv(os.path.join(submission_dir, f"lgb_feature_importance_{timestamp}.csv"), index=False)

log_message("\nTop 10 ExtraTrees feature importances:")
for i, row in et_importances.head(10).iterrows():
    log_message(f"  {row['feature']}: {row['importance']:.6f}")

log_message("\nTop 10 XGBoost feature importances:")
for i, row in xgb_importances.head(10).iterrows():
    log_message(f"  {row['feature']}: {row['importance']:.6f}")

log_message("\nTop 10 LightGBM feature importances:")
for i, row in lgb_importances.head(10).iterrows():
    log_message(f"  {row['feature']}: {row['importance']:.6f}")

end_time = time.time()
execution_time = end_time - start_time
log_message(f"\n=== Pipeline completed in {execution_time:.2f} seconds ({execution_time/60:.2f} minutes) ===")
log_message(f"Best approach: {best_approach} with validation R²: {best_score:.6f}")
log_message(f"Results saved in {submission_dir}")


=== Feature Importance Analysis ===

Top 10 ExtraTrees feature importances:
  nclimgrid_band1: 0.039678
  Income_1000m: 0.032545
  Income_500m: 0.025860
  Average_Building_Height_1000m: 0.023096
  tree_avg_diam_1000m: 0.020426
  roads_ratio_1000m: 0.020365
  nclimgrid_band4: 0.019333
  Traffic_Volume_med: 0.017134
  Average_Building_Height_750m: 0.016680
  parks_ratio_1000m: 0.016602

Top 10 XGBoost feature importances:
  Traffic_Volume_med: 0.296680
  Traffic_Volume_min: 0.091541
  disability: 0.085065
  Income_1000m: 0.034831
  Fine particles (PM 2.5): 0.029917
  _MaxTemp_ 6: 0.025002
  total_population: 0.023911
  pm2.5_std_JJA: 0.021132
  pm2.5_avg_JJA: 0.020671
  2021_07_24_00_00_2021_07_24_23_59_Sentinel_3_SLSTR_S9_(Raw): 0.020504

Top 10 LightGBM feature importances:
  dist_to_road: 2989.000000
  roads_ratio_100m: 1980.000000
  dist_to_park: 1788.000000
  tree_avg_diam_200m: 1711.000000
  tree_avg_diam_300m: 1540.000000
  tree_count_200m: 1421.000000
  roads_ratio_250m: 1384.00